In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.init import kaiming_uniform_
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # Importing StandardScaler
import numpy as np
import pandas as pd

class KANLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, grid_size=10, spline_order=3,
                 activation=None, enable_scaling=False, enable_standalone_scale_spline=False):
        super(KANLinear, self).__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.activation = activation if activation else nn.Identity()
        self.enable_scaling = enable_scaling
        self.enable_standalone_scale_spline = enable_standalone_scale_spline

        # Weights and bias
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=torch.float64))
        self.spline_weights = nn.Parameter(torch.empty(in_features, grid_size, dtype=torch.float64))  # Match grid_size
        self.spline_grid = nn.Parameter(torch.linspace(-1, 1, grid_size, dtype=torch.float64), requires_grad=False)

        if self.bias:
            self.bias = nn.Parameter(torch.empty(out_features, dtype=torch.float64))
        else:
            self.register_parameter('bias', None)

        # Optional standalone scale for spline
        if enable_standalone_scale_spline:
            self.standalone_scale_spline = nn.Parameter(torch.ones(out_features))
        else:
            self.register_parameter('standalone_scale_spline', None)

        self.reset_parameters()

    def reset_parameters(self):
        kaiming_uniform_(self.weight, a=0.0)
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / torch.sqrt(torch.tensor(fan_in, dtype=torch.float64))
            nn.init.uniform_(self.bias, -bound, bound)
        nn.init.uniform_(self.spline_weights, -0.1, 0.1)  # Uniform initialization for spline weights

    def b_splines(self, x):
        """Computes B-spline basis for input x."""
        x = x.unsqueeze(-1)  # Add a feature dimension
        grid = self.spline_grid.unsqueeze(0).unsqueeze(0)  # Reshape grid
        dists = torch.abs(x - grid)  # Distance from grid points
        basis = torch.clamp(1 - dists, min=0) ** self.spline_order
        return basis

    def forward(self, x):
        """Performs forward pass."""
        x_weighted = F.linear(x, self.weight, self.bias)  # Shape: (batch_size, out_features)

        if self.enable_scaling:
            scaled_x = x * torch.exp(self.spline_weights.mean(dim=1)).unsqueeze(0)  # Shape: (batch_size, in_features)
        else:
            scaled_x = x

        spline_basis = self.b_splines(scaled_x)  # Shape: (batch_size, in_features, grid_size)
        spline_output = torch.matmul(spline_basis, self.spline_weights.t())  # Shape: (batch_size, grid_size)
        spline_output = spline_output.mean(dim=1, keepdim=True)  # Reduce to (batch_size, out_features)

        if self.enable_standalone_scale_spline:
            spline_output = spline_output * self.standalone_scale_spline.unsqueeze(0)  # Shape: (batch_size, out_features)

        output = self.activation(x_weighted + spline_output)  # Shape: (batch_size, out_features)
        return output

    def extra_repr(self):
        return (f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias}, '
                f'grid_size={self.grid_size}, spline_order={self.spline_order}, '
                f'activation={self.activation}, enable_scaling={self.enable_scaling}, '
                f'enable_standalone_scale_spline={self.enable_standalone_scale_spline}')


# Training and Evaluation
if __name__ == "__main__":
    # Load dataset
    data = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

    # Features and target variables
    X = data[['Lat', 'Lon', 'Time', 'NO2', 'Ozone']]
    y = data['PM2.5']

    # Convert time to numerical value
    X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9

    # Normalize the features and target variable using StandardScaler
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    # Normalize X and y
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))  # y needs to be 2D

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    # Convert to tensors with consistent dtype (float64)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float64)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float64).unsqueeze(-1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float64)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float64).unsqueeze(-1)

    # Create DataLoader for mini-batch training
    batch_size = 1024  # Set a batch size based on your memory capacity
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Define model
    model = KANLinear(in_features=5, out_features=1, grid_size=20, spline_order=3,
                      activation=nn.ReLU(), enable_scaling=True, enable_standalone_scale_spline=True)

    # Loss and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    # Training loop with mini-batch processing
    epochs = 100
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            # Ensure batch tensors are of the correct dtype
            batch_X = batch_X.to(torch.float64)
            batch_y = batch_y.to(torch.float64)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss / len(train_loader):.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        predictions = model(X_test_tensor).squeeze()
        y_true = y_test_tensor.squeeze()

        # Metrics
        mse = mean_squared_error(y_true, predictions)
        mae = mean_absolute_error(y_true, predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, predictions)

        print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_415372\715431387.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9
c:\Users\Faculty\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([1024, 1, 1])) that is different to the input size (torch.Size([1024, 1024, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\Faculty\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([187, 1, 1])) 

Epoch [10/100], Loss: 1.0025
Epoch [20/100], Loss: 0.9366
Epoch [30/100], Loss: 0.9352
Epoch [40/100], Loss: 0.9351
Epoch [50/100], Loss: 0.9347
Epoch [60/100], Loss: 0.9343
Epoch [70/100], Loss: 0.9147
Epoch [80/100], Loss: 0.9057
Epoch [90/100], Loss: 0.9046
Epoch [100/100], Loss: 0.9047


RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 1173898348840 bytes.

In [34]:
print("X_train_tensor shape:", X_train_tensor.shape)  # Should be (batch_size, 5) for in_features=5


X_train_tensor shape: torch.Size([685243, 5])
